# FCSG_NET Phase 1 and Phase 2


In [4]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
DATA = Path('/kaggle/input')  # On Colab, set this to the directory containing both DIV2K splits.
REPO = BASE / 'fcsg-milestone-code'
OUT = BASE / ('fcsg-milestones-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
assert DATA.exists(), f'Attach DIV2K first: {DATA}'
import torch
assert torch.cuda.is_available(), 'Enable the GPU accelerator before running.'
print('GPU:', torch.cuda.get_device_name())
print('Results:', OUT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'thop==0.1.1.post2209072238'], check=True)


GPU: Tesla T4
Results: /kaggle/working/fcsg-milestones-20260923-102230


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'thop==0.1.1.post2209072238'], returncode=0)

## Bundled source

18 files; SHA-256 `48ef7806ff38a153be54b0a0e3374b43c97d192a833d0a9812297064cbb86985`.


In [5]:
FILES = {'src/fcsg_net/__init__.py': 'def main() -> None:\n    print("Hello from fcsg-net!")\n', 'src/fcsg_net/data.py': '"""DIV2K patch datasets.\n\nPatchDataset crops straight from PNGs. TileDataset reads a pre-extracted uint8\nmemmap built by training/build_tiles.py, which is what real runs should use:\ndecoding a 2K PNG per sample caps the loader at roughly 1.8 steps/s.\n\nThese datasets return clean crops. DegradedDataset wraps either dataset for\ncomposite degradation in DataLoader workers; Gaussian noise stays on the GPU.\n"""\n\nimport argparse\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nIMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}\n\n\ndef find_images(root):\n    """All images under root, sorted. Recurses, so a dataset dir or its parent works."""\n    root = Path(root)\n    files = sorted(p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS)\n    if not files:\n        raise FileNotFoundError(f"no images under {root}")\n    return files\n\n\ndef _augment(t, rng):\n    if rng.random() < 0.5:\n        t = torch.flip(t, dims=[2])\n    k = rng.randint(0, 3)\n    if k:\n        t = torch.rot90(t, k, dims=[1, 2])\n    return t.contiguous()\n\n\ndef _to_tensor(a):\n    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div(255.0)\n\n\ndef ensure_min_size(img, p):\n    """Upscale proportionally if either side is under p, so crops never distort."""\n    w, h = img.size\n    if w < p or h < p:\n        s = p / min(w, h)\n        img = img.resize((max(p, round(w * s)), max(p, round(h * s))), Image.BICUBIC)\n    return img\n\n\nclass PatchDataset(Dataset):\n    """Random crops with flip and 90-degree rotation augmentation.\n\n    seed=None gives fresh randomness every epoch (training). An int seed makes\n    the crop for a given index identical across runs and machines, which is what\n    validation needs for numbers to be comparable between sessions.\n    """\n\n    def __init__(self, root, patch=128, seed=None, limit=None):\n        self.files = find_images(root)\n        if limit is not None:\n            self.files = self.files[:limit]\n        self.patch = patch\n        self.seed = seed\n\n    def __len__(self):\n        return len(self.files)\n\n    def __getitem__(self, idx):\n        rng = random.Random(self.seed + idx) if self.seed is not None else random\n        p = self.patch\n        img = ensure_min_size(Image.open(self.files[idx]).convert("RGB"), p)\n        w, h = img.size\n        x, y = rng.randint(0, w - p), rng.randint(0, h - p)\n        t = _to_tensor(np.array(img.crop((x, y, x + p, y + p))))\n        return {"hr": _augment(t, rng)}\n\n\nclass TileDataset(Dataset):\n    """Pre-extracted tiles from a .npy memmap, shape (N, patch, patch, 3) uint8."""\n\n    def __init__(self, path, seed=None):\n        self.path = Path(path)\n        self.seed = seed\n        self.tiles = None\n        self.n = len(np.load(self.path, mmap_mode="r"))\n\n    def __len__(self):\n        return self.n\n\n    def __getitem__(self, idx):\n        if self.tiles is None:\n            # Opened on first access so every DataLoader worker maps its own handle.\n            self.tiles = np.load(self.path, mmap_mode="r")\n        rng = random.Random(self.seed + idx) if self.seed is not None else random\n        return {"hr": _augment(_to_tensor(self.tiles[idx]), rng)}\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="render a crop grid to eyeball the pipeline")\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out", default="data_check.png")\n    ap.add_argument("--patch", type=int, default=128)\n    ap.add_argument("--n", type=int, default=8)\n    ap.add_argument("--tiles", action="store_true", help="--data points at a .npy tile cache")\n    args = ap.parse_args()\n\n    from torchvision.utils import save_image\n\n    ds = TileDataset(args.data, seed=0) if args.tiles else PatchDataset(args.data, patch=args.patch, seed=0)\n    print(f"{len(ds)} samples in {args.data}")\n    batch = torch.stack([ds[i]["hr"] for i in range(args.n)])\n    assert batch.shape[0] == args.n and batch.shape[1] == 3, batch.shape\n    assert 0.0 <= batch.min() and batch.max() <= 1.0\n    assert batch.std() > 0.02, "crops are near-constant, check decoding"\n    save_image(batch, args.out, nrow=4)\n    print(f"wrote {args.out}  mean={batch.mean():.3f} std={batch.std():.3f}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/fcsg_net/degrade.py': '"""On-the-fly archival degradation and a reproducible D2 snapshot."""\n\nimport argparse\nimport csv\nimport io\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageFilter\nfrom torch.utils.data import Dataset\n\n\ndef degrade(hr, rng, resample=True):\n    """Return an RGB tensor and the sampled parameters. Input is CPU CHW [0, 1]."""\n    if hr.ndim != 3 or hr.shape[0] != 3 or min(hr.shape[-2:]) < 2:\n        raise ValueError("expected an RGB CHW image at least 2x2")\n    if hr.device.type != "cpu" or not torch.isfinite(hr).all() or hr.min() < 0 or hr.max() > 1:\n        raise ValueError("degradation expects a finite CPU image in [0, 1]")\n    params = {"blur_sigma": float(rng.uniform(0.5, 2.0)),\n              "scale": float(rng.uniform(1, 2)) if resample else 1.0,\n              "noise_sigma": float(rng.uniform(5, 50)),\n              "jpeg_quality": int(rng.integers(30, 96))}\n    a = hr.permute(1, 2, 0).numpy()\n    img = Image.fromarray(np.rint(a * 255).astype(np.uint8))\n    img = img.filter(ImageFilter.GaussianBlur(params["blur_sigma"]))\n    if resample:\n        size = img.size\n        small = tuple(max(1, round(s / params["scale"])) for s in size)\n        img = img.resize(small, Image.Resampling.BICUBIC).resize(size, Image.Resampling.BICUBIC)\n    a = np.asarray(img, dtype=np.float32) / 255\n    a = np.clip(a + rng.normal(0, params["noise_sigma"] / 255, a.shape), 0, 1)\n    buffer = io.BytesIO()\n    Image.fromarray(np.rint(a * 255).astype(np.uint8)).save(\n        buffer, format="JPEG", quality=params["jpeg_quality"], subsampling=0)\n    buffer.seek(0)\n    with Image.open(buffer) as decoded:\n        a = np.array(decoded.convert("RGB"), copy=True)\n    return torch.from_numpy(a).permute(2, 0, 1).float() / 255, params\n\n\nclass DegradedDataset(Dataset):\n    def __init__(self, clean, seed=None, resample=True):\n        self.clean, self.seed, self.resample = clean, seed, resample\n\n    def __len__(self):\n        return len(self.clean)\n\n    def __getitem__(self, idx):\n        sample = self.clean[idx]\n        # DataLoader seeds Python random per worker; an explicit seed fixes validation.\n        seed = self.seed + idx if self.seed is not None else random.getrandbits(63)\n        lr, params = degrade(sample["hr"], np.random.default_rng(seed), self.resample)\n        return {**sample, "lr": lr, **params}\n\n\ndef snapshot(data, out, count=5000, seed=1234, patch=128):\n    from fcsg_net.data import PatchDataset\n\n    if count < 1 or patch < 2:\n        raise ValueError("count must be positive and patch >= 2")\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=False)\n    (out / "hr").mkdir()\n    (out / "lr").mkdir()\n    clean = PatchDataset(data, patch=patch, seed=seed)\n    t0 = time.perf_counter()\n    with (out / "manifest.csv").open("w", newline="") as f:\n        fields = ["id", "source", "crop_seed", "noise_seed", "blur_sigma", "scale", "noise_sigma", "jpeg_quality"]\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        for i in range(count):\n            # Fresh deterministic crop even after cycling through the source images.\n            clean.seed = seed + i * len(clean)\n            hr = clean[i % len(clean)]["hr"]\n            lr, params = degrade(hr, np.random.default_rng(seed + i))\n            for name, tensor in (("hr", hr), ("lr", lr)):\n                a = tensor.permute(1, 2, 0).numpy()\n                Image.fromarray(np.rint(a * 255).astype(np.uint8)).save(out / name / f"{i:05d}.png")\n            writer.writerow({"id": i, "source": str(clean.files[i % len(clean)].relative_to(data)),\n                             "crop_seed": clean.seed + i % len(clean), "noise_seed": seed + i, **params})\n            if (i + 1) % 250 == 0:\n                print(f"D2 snapshot {i + 1}/{count}", flush=True)\n    return {"pairs": count, "seed": seed, "patch": patch,\n            "seconds": time.perf_counter() - t0, "path": str(out)}\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="training HR images only")\n    ap.add_argument("--out", required=True, help="new directory for 5,000 PNG pairs and metadata")\n    ap.add_argument("--count", type=int, default=5000)\n    ap.add_argument("--seed", type=int, default=1234)\n    args = ap.parse_args()\n    print(snapshot(args.data, args.out, args.count, args.seed))\n', 'src/fcsg_net/metrics.py': '"""Metrics."""\n\nimport torch\n\n\ndef psnr(pred, target, max_val=1.0, clip=True):\n    """Mean RGB PSNR. Set clip=False to measure an unclipped noisy input."""\n    pred = (pred.clamp(0, max_val) if clip else pred).float()\n    target = target.clamp(0, max_val).float()\n    mse = ((pred - target) ** 2).flatten(1).mean(1)\n    return (10 * torch.log10(max_val**2 / mse.clamp_min(1e-12))).mean().item()\n', 'src/fcsg_net/utils.py': '"""Shared helpers: degradation for Phase 1, checkpointing, CSV logging."""\n\nimport csv\nimport os\nimport re\nfrom pathlib import Path\n\nimport torch\n\n\ndef add_noise(x, sigma, generator=None, clip=True):\n    """Additive white Gaussian noise. sigma is in [0, 1] units, i.e. 25/255."""\n    noise = torch.randn(x.shape, device=x.device, dtype=x.dtype, generator=generator)\n    noisy = x + noise * sigma\n    return noisy.clamp(0, 1) if clip else noisy\n\n\ndef save_checkpoint(path, step, model, optimizer=None, scaler=None, extra=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "step": step,\n            "model": model.state_dict(),\n            "optimizer": optimizer.state_dict() if optimizer else None,\n            "scaler": scaler.state_dict() if scaler else None,\n            "extra": extra or {},\n        },\n        path,\n    )\n\n\ndef newest_checkpoint(out_dir):\n    """Highest-step ckpt_*.pt in out_dir, or None. Sorts numerically, not lexically."""\n    ckpts = list(Path(out_dir).glob("ckpt_*.pt"))\n    if not ckpts:\n        return None\n    return max(ckpts, key=lambda p: int(re.search(r"(\\d+)", p.stem).group(1)))\n\n\ndef load_checkpoint(path, model, optimizer=None, scaler=None, device="cpu"):\n    """Restore state and return the step to resume from."""\n    ck = torch.load(path, map_location=device, weights_only=False)\n    model.load_state_dict(ck["model"])\n    if optimizer and ck.get("optimizer"):\n        optimizer.load_state_dict(ck["optimizer"])\n    if scaler and ck.get("scaler"):\n        scaler.load_state_dict(ck["scaler"])\n    return ck["step"]\n\n\nclass CSVLog:\n    """Append-only CSV. stdout does not survive a killed Kaggle session; this does."""\n\n    def __init__(self, path, fields):\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.fields = fields\n        if not self.path.exists():\n            with self.path.open("w", newline="") as f:\n                csv.writer(f).writerow(fields)\n\n    def write(self, **row):\n        with self.path.open("a", newline="") as f:\n            csv.writer(f).writerow([row.get(k, "") for k in self.fields])\n\n\nIMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}\n\n\ndef image_dirs(root, limit=50000):\n    """Every directory under root that directly holds images, with its count."""\n    found = {}\n    for dirpath, _, filenames in os.walk(root):\n        n = sum(1 for f in filenames if os.path.splitext(f)[1].lower() in IMG_EXTS)\n        if n:\n            found[Path(dirpath)] = n\n        if len(found) >= limit:\n            break\n    return found\n\n\ndef _pick(dirs, keywords, taken=None):\n    """First directory whose name matches a keyword, keywords in priority order."""\n    for kw in keywords:\n        for d in sorted(dirs):\n            if d in (taken or ()):\n                continue\n            if kw in d.name.lower():\n                return d\n    return None\n\n\ndef resolve_div2k(root, train_dir=None, val_dir=None):\n    """Find the training and validation image directories under root.\n\n    Kaggle dataset layouts differ from each other and from a Colab unzip, so\n    nothing hardcodes a path: point --data at /kaggle/input and let it search.\n    Pass train_dir and val_dir explicitly to skip the search entirely.\n    """\n    if train_dir and val_dir:\n        return Path(train_dir), Path(val_dir)\n\n    root = Path(root)\n    if not root.exists():\n        raise FileNotFoundError(root)\n\n    dirs = image_dirs(root)\n    if not dirs:\n        raise FileNotFoundError(\n            f"no images anywhere under {root}. Attach a DIV2K dataset in the "\n            f"Kaggle sidebar before running this."\n        )\n\n    train = Path(train_dir) if train_dir else _pick(dirs, ("train_hr", "train"))\n    val = Path(val_dir) if val_dir else _pick(dirs, ("valid_hr", "valid", "val", "test"), taken={train})\n\n    if train is None and val is not None:\n        train = _pick(dirs, ("hr", ""), taken={val})\n    if train is not None and val is None and len(dirs) == 1:\n        val = train\n        print(f"WARNING: only one image directory found ({train}). Validating on "\n              f"training images. Attach a dataset with a validation split before "\n              f"you trust any PSNR number.")\n\n    if train is None or val is None:\n        listing = "\\n".join(f"  {d}  ({n} images)" for d, n in sorted(dirs.items())[:20])\n        raise FileNotFoundError(\n            f"could not identify train and validation directories under {root}.\\n"\n            f"Directories holding images:\\n{listing}\\n"\n            f"Pass --train-dir and --val-dir explicitly with two of these paths."\n        )\n    return train, val\n', 'models/blocks.py': '"""Frequency bands, shared experts, spatial routing, and SE fusion."""\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass FrequencyDecompose(nn.Module):\n    def __init__(self, low=0.10, high=0.25, transition=0.05):\n        super().__init__()\n        if not (0 < transition < min(2 * low, high - low) and low < high < 0.5):\n            raise ValueError("require 0 < low < high < 0.5 and nonoverlapping transitions")\n        self.low, self.high, self.transition = low, high, transition\n\n    def masks(self, h, w, device):\n        # fftfreq is already in the unshifted ordering used by rfft2.\n        fy = torch.fft.fftfreq(h, device=device)[:, None]\n        fx = torch.fft.rfftfreq(w, device=device)[None, :]\n        radius = torch.sqrt(fy.square() + fx.square())\n        def lowpass(cutoff):\n            t = ((radius - cutoff) / self.transition + 0.5).clamp(0, 1)\n            return 0.5 * (1 + torch.cos(math.pi * t))\n        low, upper = lowpass(self.low), lowpass(self.high)\n        return torch.stack((low, upper - low, 1 - upper))\n\n    def forward(self, x):\n        # CUDA half FFTs require powers of two. Float32 also preserves the sum.\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            spectrum = torch.fft.rfft2(x.float(), norm="ortho")\n            masks = self.masks(*x.shape[-2:], x.device)\n            bands = torch.fft.irfft2(spectrum[:, None] * masks[None, :, None],\n                                    s=x.shape[-2:], norm="ortho")\n        return bands\n\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, width, kernel):\n        super().__init__()\n        self.body = nn.Sequential(\n            nn.Conv2d(width, width, kernel, padding=kernel // 2, groups=width),\n            nn.Conv2d(width, width, 1), nn.ReLU(),\n            nn.Conv2d(width, width, kernel, padding=kernel // 2, groups=width),\n            nn.Conv2d(width, width, 1),\n        )\n\n    def forward(self, x):\n        return x + self.body(x)\n\n\nclass Expert(nn.Module):\n    def __init__(self, kernel, width=48, blocks=4):\n        super().__init__()\n        # Lossless rearrangement and separable convs target the 10 GFLOP budget.\n        self.head = nn.Conv2d(12, width, 1)\n        self.body = nn.Sequential(*(ResidualBlock(width, kernel) for _ in range(blocks)))\n        self.tail = nn.Conv2d(width, 12, 1)\n\n    def forward(self, x):\n        h, w = x.shape[-2:]\n        z = F.pixel_unshuffle(F.pad(x, (0, w % 2, 0, h % 2), mode="replicate"), 2)\n        z = z + self.tail(self.body(F.relu(self.head(z))))\n        return F.pixel_shuffle(z, 2)[..., :h, :w]\n\n\nclass Gate(nn.Module):\n    def __init__(self, window=16, tau=1.0, top_k=2):\n        super().__init__()\n        if window < 1 or tau <= 0 or not 1 <= top_k <= 3:\n            raise ValueError("invalid gate window, temperature, or top_k")\n        self.window, self.tau, self.top_k = window, tau, top_k\n        self.net = nn.Sequential(nn.Conv2d(18, 24, 1), nn.ReLU(), nn.Conv2d(24, 9, 1))\n\n    def forward(self, bands):\n        b, _, _, h, w = bands.shape\n        z = bands.reshape(b, 9, h, w)\n        window = min(self.window, h, w)\n        mean = F.avg_pool2d(z, window, window, ceil_mode=True)\n        energy = F.avg_pool2d(z.square(), window, window, ceil_mode=True)\n        logits = self.net(torch.cat((mean, energy), 1)) / self.tau\n        logits = F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)\n        logits = logits.reshape(b, 3, 3, h, w)\n        dense = logits.softmax(dim=2)\n        values, indices = logits.topk(self.top_k, dim=2)\n        sparse = torch.zeros_like(logits).scatter(2, indices, values.softmax(dim=2))\n        return sparse, dense\n\n\nclass SEFusion(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.attention = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(9, 3, 1),\n                                       nn.ReLU(), nn.Conv2d(3, 9, 1), nn.Sigmoid())\n        self.project = nn.Conv2d(9, 3, 1)\n\n    def forward(self, x):\n        return self.project(x * self.attention(x))\n', 'models/dncnn.py': '"""DnCNN (Zhang et al., TIP 2017)."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass DnCNN(nn.Module):\n    def __init__(self, channels=3, depth=17, width=64):\n        super().__init__()\n        layers = [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=True)]\n        for _ in range(depth - 2):\n            layers += [\n                nn.Conv2d(width, width, 3, padding=1, bias=False),\n                nn.BatchNorm2d(width),\n                nn.ReLU(inplace=True),\n            ]\n        layers += [nn.Conv2d(width, channels, 3, padding=1)]\n        self.body = nn.Sequential(*layers)\n\n    def forward(self, x):\n        return x - self.body(x)\n\n\ndef build(**kwargs):\n    return DnCNN(**kwargs)\n\n\nif __name__ == "__main__":\n    m = DnCNN()\n    n = sum(p.numel() for p in m.parameters())\n    print(f"DnCNN params: {n:,}")\n    # 17 layers at width 64 is ~0.56M. A silent width or depth typo shows up here\n    # rather than after a ten-hour run.\n    assert 550_000 < n < 570_000, n\n    x = torch.rand(2, 3, 64, 64)\n    y = m(x)\n    assert y.shape == x.shape, y.shape\n    assert torch.isfinite(y).all()\n    print("dncnn ok")\n', 'models/fcsg.py': '"""Frequency selection followed by shared spatial experts and sparse mixing."""\n\nimport torch\nfrom torch import nn\n\nfrom models.blocks import Expert, FrequencyDecompose, Gate, SEFusion\n\n\nclass FCSGNet(nn.Module):\n    def __init__(self, width=48, blocks=4, tau=1.0, top_k=2):\n        super().__init__()\n        if width < 1 or blocks < 1:\n            raise ValueError("width and blocks must be positive")\n        self.decompose = FrequencyDecompose()\n        self.experts = nn.ModuleList(Expert(k, width, blocks) for k in (7, 5, 3))\n        self.gate = Gate(tau=tau, top_k=top_k)\n        self.fusion = SEFusion()\n        self.refine = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),\n                                    nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),\n                                    nn.Conv2d(16, 3, 3, padding=1))\n\n    def forward(self, x, return_aux=False):\n        bands = self.decompose(x)\n        routing, dense = self.gate(bands)\n        b, _, c, h, w = bands.shape\n        flat = bands.reshape(b * 3, c, h, w)\n        mixed = torch.zeros_like(bands)\n        # ponytail: all experts execute; sparse dispatch needs tiled expert execution.\n        for e, expert in enumerate(self.experts):\n            mixed = mixed + expert(flat).reshape(b, 3, c, h, w) * routing[:, :, e, None]\n        out = x + self.refine(self.fusion(mixed.reshape(b, 9, h, w)))\n        if return_aux:\n            return out, {"bands": bands, "routing": routing, "dense_routing": dense,\n                         "band_outputs": mixed}\n        return out\n', 'models/ffdnet.py': '"""Colour FFDNet, compatible with the author\'s KAIR ffdnet_color.pth weights.\n\nReference: https://github.com/cszn/KAIR/blob/master/models/network_ffdnet.py\nThe author released inference weights with batch normalization folded into convs.\n"""\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass FFDNet(nn.Module):\n    def __init__(self, channels=3, depth=12, width=96):\n        super().__init__()\n        if channels != 3 or depth < 2 or width < 1:\n            raise ValueError("FFDNet expects RGB, depth >= 2, and positive width")\n        layers = [nn.Conv2d(channels * 4 + 1, width, 3, padding=1), nn.ReLU(True)]\n        for _ in range(depth - 2):\n            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU(True)]\n        layers.append(nn.Conv2d(width, channels * 4, 3, padding=1))\n        self.model = nn.Sequential(*layers)\n\n    def forward(self, x, sigma):\n        """sigma is in [0, 1] units: scalar, (B,), or (B, 1, H, W)."""\n        h, w = x.shape[-2:]\n        z = F.pixel_unshuffle(F.pad(x, (0, w % 2, 0, h % 2), mode="replicate"), 2)\n        sigma = torch.as_tensor(sigma, device=x.device, dtype=x.dtype)\n        if sigma.ndim <= 1:\n            sigma = sigma.reshape(-1, 1, 1, 1)\n        if sigma.ndim != 4 or sigma.shape[1] != 1 or sigma.shape[0] not in (1, x.shape[0]):\n            raise ValueError("sigma must be scalar, per-image, or a single-channel noise map")\n        if not torch.isfinite(sigma).all() or (sigma < 0).any() or (sigma > 1).any():\n            raise ValueError("sigma must be finite and between 0 and 1")\n        sigma = F.interpolate(sigma, size=z.shape[-2:], mode="bilinear", align_corners=False)\n        z = self.model(torch.cat((z, sigma.expand(x.shape[0], -1, -1, -1)), dim=1))\n        return F.pixel_shuffle(z, 2)[..., :h, :w]\n', 'training/build_tiles.py': '"""Pre-extract DIV2K crops into a uint8 .npy memmap.\n\nDecoding a 2K PNG for every training sample caps the loader at roughly\n1.8 steps/s, which is a 46-hour run at 300k steps. Tiles come off a memmap\ninstead, so the bottleneck moves back to the GPU. Run once per patch size.\n\n    python training/build_tiles.py --data /kaggle/input --out /kaggle/working/tiles.npy\n"""\n\nimport argparse\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path[:0] = [str(ROOT), str(ROOT / "src")]\n\nfrom fcsg_net.data import ensure_min_size, find_images  # noqa: E402\nfrom fcsg_net.utils import resolve_div2k  # noqa: E402\n\n\ndef build(files, out, patch, per_image, seed=0):\n    rng = random.Random(seed)\n    out = Path(out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    n = len(files) * per_image\n    tiles = np.lib.format.open_memmap(\n        out, mode="w+", dtype=np.uint8, shape=(n, patch, patch, 3)\n    )\n    print(f"{len(files)} images x {per_image} crops = {n} tiles "\n          f"({tiles.nbytes / 1e9:.2f} GB) -> {out}")\n\n    t0 = time.time()\n    for i, f in enumerate(files):\n        img = ensure_min_size(Image.open(f).convert("RGB"), patch)\n        a = np.asarray(img)\n        h, w = a.shape[:2]\n        for j in range(per_image):\n            y, x = rng.randint(0, h - patch), rng.randint(0, w - patch)\n            tiles[i * per_image + j] = a[y : y + patch, x : x + patch]\n        if (i + 1) % 50 == 0 or i + 1 == len(files):\n            print(f"  {i + 1}/{len(files)} images  {time.time() - t0:.0f}s")\n    tiles.flush()\n    return tiles\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="any ancestor of the DIV2K image directories")\n    ap.add_argument("--train-dir", help="skip the search and use this directory")\n    ap.add_argument("--val-dir")\n    ap.add_argument("--out", required=True, help="path to the .npy to write")\n    ap.add_argument("--patch", type=int, default=128)\n    ap.add_argument("--per-image", type=int, default=32)\n    ap.add_argument("--seed", type=int, default=0)\n    args = ap.parse_args()\n\n    train_dir, _ = resolve_div2k(args.data, args.train_dir, args.val_dir)\n    files = find_images(train_dir)\n    print(f"train={train_dir}")\n    tiles = build(files, args.out, args.patch, args.per_image, args.seed)\n\n    sample = np.asarray(tiles[:: max(len(tiles) // 256, 1)], dtype=np.float32)\n    assert sample.std() > 5.0, "tiles are near-constant, check decoding"\n    print(f"done  mean={sample.mean():.1f}  std={sample.std():.1f}")\n\n\nif __name__ == "__main__":\n    main()\n', 'training/train.py': '"""One config-driven training entry point for every model in the project.\n\nPhase 4 ablations become config diffs rather than forked scripts, which is also\nhow you keep them honest.\n\n    python training/train.py --config configs/dncnn.toml --data /kaggle/input \\\n        --out /kaggle/working/ckpt\n"""\n\nimport argparse\nimport math\nimport sys\nimport time\nimport tomllib\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path[:0] = [str(ROOT), str(ROOT / "src")]\n\nfrom fcsg_net.data import PatchDataset, TileDataset  # noqa: E402\nfrom fcsg_net.degrade import DegradedDataset  # noqa: E402\nfrom fcsg_net.metrics import psnr  # noqa: E402\nfrom fcsg_net.utils import (  # noqa: E402\n    CSVLog,\n    add_noise,\n    load_checkpoint,\n    newest_checkpoint,\n    resolve_div2k,\n    save_checkpoint,\n)\n\nVAL_SEED = 1234\n\n\ndef build_model(name, **kwargs):\n    if name == "dncnn":\n        from models.dncnn import DnCNN\n\n        return DnCNN(**kwargs)\n    if name == "ffdnet":\n        from models.ffdnet import FFDNet\n\n        return FFDNet(**kwargs)\n    if name == "fcsg":\n        from models.fcsg import FCSGNet\n\n        return FCSGNet(**kwargs)\n    raise ValueError(f"unknown model {name!r}")\n\n\ndef predict(model, noisy, sigma):\n    from models.ffdnet import FFDNet\n\n    return model(noisy, sigma) if isinstance(model, FFDNet) else model(noisy)\n\n\ndef charbonnier(pred, target, eps=1e-3):\n    """Standard restoration loss. Used for the baselines too, so the Phase 3\n    comparison against FCSG-Net is like for like."""\n    return torch.sqrt((pred - target) ** 2 + eps**2).mean()\n\n\ndef cosine_lr(step, total, lr, lr_min):\n    """Computed from the step, so resuming needs no scheduler state."""\n    t = min(step / max(total, 1), 1.0)\n    return lr_min + 0.5 * (lr - lr_min) * (1 + math.cos(math.pi * t))\n\n\ndef infinite(loader):\n    while True:\n        yield from loader\n\n\n@torch.no_grad()\ndef validate(model, val_batch, sigma, device, clip=True):\n    """Fixed crops, fixed noise, so numbers are comparable across sessions."""\n    model.eval()\n    hr = val_batch["hr"].to(device)\n    g = torch.Generator(device=device).manual_seed(VAL_SEED)\n    noisy = val_batch["lr"].to(device) if "lr" in val_batch else add_noise(hr, sigma, generator=g, clip=clip)\n    levels = val_batch["noise_sigma"].to(device) / 255 if "lr" in val_batch else sigma\n    out = predict(model, noisy, levels)\n    model.train()\n    return psnr(noisy, hr, clip=False), psnr(out, hr)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", required=True)\n    ap.add_argument("--data", required=True, help="any ancestor of the DIV2K image directories")\n    ap.add_argument("--train-dir", help="skip the search and use this directory for training")\n    ap.add_argument("--val-dir", help="skip the search and use this directory for validation")\n    ap.add_argument("--tiles", help="a .npy tile cache from build_tiles.py; much faster than --data")\n    ap.add_argument("--out", required=True, help="checkpoints and log land here")\n    ap.add_argument("--steps", type=int, help="override config, for smoke runs")\n    ap.add_argument("--batch", type=int)\n    ap.add_argument("--overfit-one-image", action="store_true")\n    ap.add_argument("--max-hours", type=float, help="override config, stop and save before this")\n    args = ap.parse_args()\n\n    with open(args.config, "rb") as f:\n        cfg = tomllib.load(f)\n    for k in ("steps", "batch", "max_hours"):\n        if getattr(args, k) is not None:\n            cfg[k] = getattr(args, k)\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    print(f"device={device} config={args.config} steps={cfg[\'steps\']} batch={cfg[\'batch\']}")\n    if device == "cpu":\n        print("WARNING: no GPU visible. Enable the accelerator before a real run.")\n\n    train_dir, val_dir = resolve_div2k(args.data, args.train_dir, args.val_dir)\n    print(f"train={train_dir}\\nval={val_dir}")\n\n    sigma = cfg["sigma"] / 255.0\n    if cfg.get("degradation", "gaussian") not in ("gaussian", "composite"):\n        raise ValueError("degradation must be \'gaussian\' or \'composite\'")\n    composite = cfg.get("degradation", "gaussian") == "composite"\n    clip = cfg.get("clip_noise", True)\n    if args.overfit_one_image:\n        # One fixed crop with one fixed noise realisation, repeated forever. A\n        # model that cannot memorise this has a bug, and finding out costs two\n        # minutes here instead of ten hours on a real run.\n        one = PatchDataset(train_dir, patch=cfg["patch"], seed=0, limit=1)\n        one = DegradedDataset(one, seed=0) if composite else one\n        train_batch = next(iter(DataLoader(one, batch_size=1)))\n        loader = None\n    else:\n        workers = cfg.get("workers", 4)\n        tiles = args.tiles or cfg.get("tiles")\n        train_ds = TileDataset(tiles) if tiles else PatchDataset(train_dir, patch=cfg["patch"])\n        if composite:\n            train_ds = DegradedDataset(train_ds)\n        if len(train_ds) < cfg["batch"]:\n            raise ValueError("training dataset is smaller than the batch size")\n        print(f"train samples={len(train_ds)} source={tiles or train_dir}")\n        loader = DataLoader(\n            train_ds,\n            batch_size=cfg["batch"],\n            shuffle=True,\n            num_workers=workers,\n            drop_last=True,\n            pin_memory=True,\n            persistent_workers=workers > 0,\n        )\n\n    val_ds = PatchDataset(val_dir, patch=cfg["patch"], seed=VAL_SEED, limit=cfg.get("val_images", 16))\n    if composite:\n        val_ds = DegradedDataset(val_ds, seed=VAL_SEED)\n    val_batch = next(iter(DataLoader(val_ds, batch_size=len(val_ds))))\n\n    model = build_model(cfg["model"], **cfg.get("model_args", {})).to(device)\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f"model={cfg[\'model\']} params={n_params:,}")\n\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 0.0))\n    # float16, not bfloat16: neither the P100 nor the T4 supports bf16.\n    use_amp = device == "cuda" and cfg.get("amp", True)\n    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)\n\n    out_dir = Path(args.out)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    start = 0\n    latest = newest_checkpoint(out_dir)\n    if latest:\n        start = load_checkpoint(latest, model, opt, scaler, device=device)\n        print(f"RESUMED from {latest.name} at step {start}")\n    else:\n        print("no checkpoint found, starting from step 0")\n\n    log = CSVLog(out_dir / "train_log.csv", ["step", "loss", "lr", "psnr_in", "psnr_out", "secs"])\n    total = cfg["steps"]\n    # A Kaggle session killed at the 12h cap loses /kaggle/working entirely, so\n    # the run has to stop itself with time to spare and let the notebook finish.\n    budget = cfg.get("max_hours", 11.5) * 3600\n    if start >= total:\n        print(f"already at step {start}/{total}, nothing to do")\n        return\n    data = infinite(loader) if loader is not None else None\n    fixed_noisy = None\n    t0 = time.time()\n\n    for step in range(start + 1, total + 1):\n        lr = cosine_lr(step, total, cfg["lr"], cfg["lr_min"])\n        for g in opt.param_groups:\n            g["lr"] = lr\n\n        if loader is None:\n            hr = train_batch["hr"].to(device)\n            if fixed_noisy is None:\n                g = torch.Generator(device=device).manual_seed(0)\n                fixed_noisy = train_batch["lr"].to(device) if composite else add_noise(hr, sigma, generator=g, clip=clip)\n            noisy = fixed_noisy\n            levels = train_batch["noise_sigma"].to(device) / 255 if composite else sigma\n        else:\n            sample = next(data)\n            hr = sample["hr"].to(device, non_blocking=True)\n            noisy = sample["lr"].to(device, non_blocking=True) if composite else add_noise(hr, sigma, clip=clip)\n            levels = sample["noise_sigma"].to(device) / 255 if composite else sigma\n\n        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=use_amp):\n            loss = charbonnier(predict(model, noisy, levels), hr)\n\n        opt.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.step(opt)\n        scaler.update()\n\n        if step % cfg.get("log_every", 50) == 0 or step == total:\n            print(f"step {step}/{total}  loss {loss.item():.5f}  lr {lr:.2e}  {time.time() - t0:.0f}s")\n            log.write(step=step, loss=f"{loss.item():.6f}", lr=f"{lr:.3e}", secs=f"{time.time() - t0:.0f}")\n\n        if step % cfg.get("val_every", 2000) == 0 or step == total:\n            p_in, p_out = validate(model, val_batch, sigma, device, clip=clip)\n            print(f"  val psnr  in {p_in:.2f} dB  ->  out {p_out:.2f} dB")\n            log.write(step=step, psnr_in=f"{p_in:.3f}", psnr_out=f"{p_out:.3f}", secs=f"{time.time() - t0:.0f}")\n\n        out_of_time = time.time() - t0 > budget\n        if step % cfg.get("ckpt_every", 2000) == 0 or step == total or out_of_time:\n            save_checkpoint(out_dir / f"ckpt_{step:07d}.pt", step, model, opt, scaler,\n                            extra={"config": cfg, "params": n_params})\n            print(f"  saved ckpt_{step:07d}.pt")\n\n        if out_of_time:\n            rate = (step - start) / max(time.time() - t0, 1)\n            print(f"stopping at step {step}/{total}: {budget / 3600:.1f}h budget reached. "\n                  f"Re-run the notebook with this output attached to resume. "\n                  f"~{(total - step) / max(rate, 1e-9) / 3600:.1f}h of GPU left at this rate.")\n            break\n\n    print(f"stopped at step {step}/{total} in {time.time() - t0:.0f}s")\n\n\nif __name__ == "__main__":\n    main()\n', 'evaluation/checks.py': '"""Executable Phase 1/2 checks. Run on Kaggle/Colab, before long training."""\n\nimport argparse\nimport json\nimport math\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path[:0] = [str(ROOT), str(ROOT / "src")]\n\nfrom fcsg_net.data import PatchDataset, TileDataset\nfrom fcsg_net.degrade import DegradedDataset, degrade, snapshot\nfrom fcsg_net.metrics import psnr\nfrom fcsg_net.utils import add_noise, load_checkpoint, save_checkpoint\nfrom models.blocks import FrequencyDecompose\nfrom models.fcsg import FCSGNet\nfrom models.ffdnet import FFDNet\nfrom training.train import build_model, charbonnier, predict, validate\nfrom evaluation.eval import tiled_forward\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True, help="training HR directory")\n    ap.add_argument("--out", default="results/phase2_checks.json")\n    ap.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    args = ap.parse_args()\n    torch.manual_seed(7)\n    torch.set_num_threads(2)\n    device = args.device\n    if device == "cuda":\n        if not torch.cuda.is_available():\n            raise RuntimeError("GPU checks require an active CUDA runtime")\n        memory = torch.cuda.get_device_properties(0).total_memory\n        torch.cuda.set_per_process_memory_fraction(min(1, 4_000_000_000 / memory))\n    result = {"device": torch.cuda.get_device_name() if device == "cuda" else "cpu",\n              "torch": torch.__version__, "seed": 7}\n    assert abs(psnr(torch.full((1, 3, 2, 2), 2.0), torch.zeros(1, 3, 2, 2), clip=False)\n               + 20 * math.log10(2)) < 1e-5\n\n    split = FrequencyDecompose().to(device)\n    errors = []\n    for shape in ((1, 3, 32, 48), (1, 3, 65, 67), (1, 3, 128, 128), (1, 3, 256, 256)):\n        x = torch.rand(shape, device=device)\n        bands = split(x)\n        errors.append((bands.sum(1) - x).abs().max().item())\n        assert torch.allclose(bands.sum(1), x, atol=1e-5), shape\n        assert split.masks(*shape[-2:], device).min() >= 0\n    # Same cycles/pixel, different FFT bin index. Low/mid/high pure tones.\n    for band, freq in enumerate((1 / 16, 3 / 16, 3 / 8)):\n        energies = []\n        for size in (128, 256):\n            tone = torch.cos(2 * math.pi * freq * torch.arange(size, device=device))\n            x = tone[None, None, None].expand(1, 3, size, size)\n            energies.append(split(x).square().mean((0, 2, 3, 4)))\n        assert torch.allclose(energies[0], energies[1], atol=1e-5)\n        assert energies[0].argmax().item() == band\n    result["reconstruction_max_error"] = max(errors)\n    result["frequency_units_check"] = True\n\n    ffd = FFDNet().to(device)\n    odd = torch.rand(2, 3, 31, 35, device=device)\n    low = ffd(odd, torch.tensor([5 / 255, 50 / 255], device=device))\n    high = ffd(odd, torch.full((2, 1, 31, 35), 0.3, device=device))\n    assert low.shape == odd.shape and torch.isfinite(low).all()\n    assert not torch.allclose(low, high), "FFDNet ignores the noise map"\n    low.square().mean().backward()\n    assert ffd.model[0].weight.grad[:, -1].abs().sum() > 0\n    assert sum(p.numel() for p in ffd.parameters()) == 852108\n    try:\n        ffd(odd, -1)\n        raise AssertionError("negative noise level accepted")\n    except ValueError:\n        pass\n    result["ffdnet_noise_map_check"] = True\n    del ffd, low, high\n\n    clean = PatchDataset(args.data, patch=128, seed=7, limit=4)\n    hr = clean[0]["hr"]\n    a, pa = degrade(hr, np.random.default_rng(7))\n    b, pb = degrade(hr, np.random.default_rng(7))\n    assert torch.equal(a, b) and pa == pb\n    assert a.shape == hr.shape and 0 <= a.min() <= a.max() <= 1\n    assert 0.5 <= pa["blur_sigma"] <= 2 and 5 <= pa["noise_sigma"] <= 50\n    assert 30 <= pa["jpeg_quality"] <= 95 and 1 <= pa["scale"] <= 2\n    assert not torch.equal(a, degrade(hr, np.random.default_rng(8))[0])\n    assert degrade(hr, np.random.default_rng(7), resample=False)[1]["scale"] == 1\n    degraded = DegradedDataset(clean, seed=7)\n    batch = next(iter(DataLoader(degraded, batch_size=2, num_workers=2)))\n    assert batch["lr"].shape == batch["hr"].shape and batch["noise_sigma"].shape == (2,)\n    assert torch.equal(batch["lr"][0], a)\n    with tempfile.TemporaryDirectory() as tmp:\n        tiles = Path(tmp) / "tiles.npy"\n        np.save(tiles, np.stack([np.rint(hr.permute(1, 2, 0).numpy() * 255).astype(np.uint8)] * 4))\n        ds = DegradedDataset(TileDataset(tiles, seed=7), seed=7)\n        assert ds[0]["lr"].shape == hr.shape\n        t0 = time.perf_counter()\n        for i in range(32):\n            ds[i % len(ds)]\n        result["degradation_samples_per_second"] = 32 / (time.perf_counter() - t0)\n        from PIL import Image\n\n        source = Path(tmp) / "source"\n        source.mkdir()\n        Image.fromarray(np.rint(hr.permute(1, 2, 0).numpy() * 255).astype(np.uint8)).save(source / "one.png")\n        first, second = Path(tmp) / "snapshot1", Path(tmp) / "snapshot2"\n        snapshot(source, first, count=2, seed=7)\n        snapshot(source, second, count=2, seed=7)\n        for path in first.rglob("*"):\n            if path.is_file():\n                assert path.read_bytes() == (second / path.relative_to(first)).read_bytes()\n    result["degradation_reproducible"] = True\n\n    model = FCSGNet().to(device)\n    x = torch.rand(1, 3, 256, 256, device=device)\n    if device == "cuda":\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n    out, aux = model(x, return_aux=True)\n    assert out.shape == x.shape and torch.isfinite(out).all()\n    weights = aux["routing"]\n    assert weights.shape == (1, 3, 3, 256, 256)\n    assert torch.allclose(weights.sum(2), torch.ones_like(weights[:, :, 0]), atol=1e-6)\n    assert ((weights > 0).sum(2) == 2).all()\n    assert weights.std(dim=(-1, -2)).max() > 0, "routing is spatially constant"\n    out.square().mean().backward()\n    assert model.gate.net[-1].weight.grad.abs().sum() > 0\n    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())\n    result["parameters"] = sum(p.numel() for p in model.parameters())\n    result["peak_allocated_bytes"] = torch.cuda.max_memory_allocated() if device == "cuda" else None\n    result["peak_reserved_bytes"] = torch.cuda.max_memory_reserved() if device == "cuda" else None\n    assert result["parameters"] < 5_000_000\n    if device == "cuda":\n        assert result["peak_reserved_bytes"] < 4_000_000_000\n    del out, aux, weights\n    model.zero_grad(set_to_none=True)\n    from thop import profile\n\n    macs, _ = profile(model, inputs=(x,), verbose=False)\n    # thop counts conv MACs, not FFTs. Conservatively charge 5*N*log2(N)\n    # real operations for each of 3 forward and 9 inverse full complex FFTs.\n    fft_ops = 12 * 5 * (256 * 256) * math.log2(256 * 256)\n    result["conv_gmacs"] = macs / 1e9\n    result["gflops_with_fft_estimate"] = (2 * macs + fft_ops) / 1e9\n    result["flop_convention"] = "2 FLOPs/MAC; thop ops plus conservative FFT estimate; elementwise ops excluded"\n    assert result["gflops_with_fft_estimate"] < 10\n    with torch.no_grad():\n        assert model(odd[:1]).shape == odd[:1].shape\n        identity = torch.nn.Identity()\n        assert torch.allclose(tiled_forward(identity, x, tile=96, overlap=16), x, atol=1e-6)\n    result["forward_backward_routing_checks"] = True\n    if device == "cuda":\n        model.zero_grad(set_to_none=True)\n        with torch.amp.autocast("cuda", dtype=torch.float16):\n            amp_out = model(odd[:1])\n            amp_loss = amp_out.square().mean()\n        amp_loss.backward()\n        assert torch.isfinite(amp_out).all()\n        assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())\n        result["amp_odd_size_check"] = True\n\n    # Exercise the shared entry points for all models on both degradation modes.\n    for name in ("dncnn", "ffdnet", "fcsg"):\n        small = build_model(name).to(device)\n        pred = predict(small, batch["lr"].to(device), batch["noise_sigma"].to(device) / 255)\n        charbonnier(pred, batch["hr"].to(device)).backward()\n        assert all(math.isfinite(v) for v in validate(small, batch, 25 / 255, device))\n        assert all(math.isfinite(v) for v in validate(small, {"hr": batch["hr"]}, 25 / 255, device, clip=False))\n        del small, pred\n\n    # 200 updates on one fixed, real image and one fixed composite degradation.\n    model = FCSGNet().to(device)\n    target, noisy = hr[None].to(device), a[None].to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0)\n    with torch.no_grad():\n        initial = (model(noisy) - target).square().mean().item()\n    losses = []\n    t0 = time.perf_counter()\n    for step in range(200):\n        optimizer.zero_grad(set_to_none=True)\n        loss = (model(noisy) - target).square().mean()\n        loss.backward()\n        optimizer.step()\n        losses.append(loss.item())\n        if (step + 1) % 50 == 0:\n            print(f"overfit {step + 1}/200 mse={loss.item():.6f}", flush=True)\n    with torch.no_grad():\n        expected = model(noisy)\n        final = (expected - target).square().mean().item()\n    result["overfit"] = {"steps": 200, "patch": 128, "initial_mse": initial,\n                         "final_mse": final, "ratio": final / initial,\n                         "seconds": time.perf_counter() - t0, "losses": losses}\n    with tempfile.TemporaryDirectory() as tmp:\n        path = Path(tmp) / "ckpt_0000200.pt"\n        save_checkpoint(path, 200, model, optimizer)\n        restored = FCSGNet().to(device)\n        opt = torch.optim.AdamW(restored.parameters())\n        assert load_checkpoint(path, restored, opt, device=device) == 200\n        with torch.no_grad():\n            assert torch.equal(restored(noisy), expected)\n        assert len(opt.state) == len(optimizer.state)\n    result["checkpoint_resume_check"] = True\n    result["overfit_passed"] = final < initial * 0.25 and final < 0.01\n    result["passed"] = result["overfit_passed"] and device == "cuda"\n    out_path = Path(args.out)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    out_path.write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps({k: v for k, v in result.items() if k != "overfit"}, indent=2))\n    print(f"overfit MSE {initial:.6f} -> {final:.6f}")\n    assert result["overfit_passed"], "200-step overfit did not reduce MSE sufficiently"\n\n\nif __name__ == "__main__":\n    main()\n', 'evaluation/eval.py': '"""Evaluate a checkpoint on full DIV2K validation images and record the result.\n\nAppends one row to results/benchmark.csv and writes the qualitative figure. The\nsame CSV is what the Phase 3 comparison table fills, so FFDNet and FCSG-Net just\nappend rows to it later.\n\n    python evaluation/eval.py --ckpt /kaggle/working/ckpt/ckpt_0000200.pt \\\n        --data /kaggle/input --out results --notes smoke\n"""\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path[:0] = [str(ROOT), str(ROOT / "src")]\n\nfrom fcsg_net.data import find_images  # noqa: E402\nfrom fcsg_net.metrics import psnr  # noqa: E402\nfrom fcsg_net.utils import CSVLog, add_noise, resolve_div2k  # noqa: E402\n\nsys.path.append(str(ROOT / "training"))\nfrom train import build_model, predict  # noqa: E402\n\n\n@torch.no_grad()\ndef tiled_forward(model, x, tile=256, overlap=32, sigma=25 / 255):\n    """Full-image inference in overlapping tiles.\n\n    FFT bands and spatial gates depend on context. Tiling is an approximation,\n    even with cutoffs in cycles/pixel. Record the tile policy with each result.\n    """\n    if not 0 <= overlap < tile:\n        raise ValueError("require 0 <= overlap < tile")\n    _, _, h, w = x.shape\n    if h <= tile and w <= tile:\n        return predict(model, x, sigma)\n    stride = tile - overlap\n    out = torch.zeros_like(x)\n    count = torch.zeros_like(x)\n    ys = list(range(0, max(h - tile, 0) + 1, stride)) or [0]\n    xs = list(range(0, max(w - tile, 0) + 1, stride)) or [0]\n    if ys[-1] + tile < h:\n        ys.append(h - tile)\n    if xs[-1] + tile < w:\n        xs.append(w - tile)\n    for y in ys:\n        for xx in xs:\n            patch = x[:, :, y : y + tile, xx : xx + tile]\n            levels = sigma\n            if isinstance(sigma, torch.Tensor) and sigma.ndim == 4 and sigma.shape[-2:] == (h, w):\n                levels = sigma[..., y : y + tile, xx : xx + tile]\n            out[:, :, y : y + tile, xx : xx + tile] += predict(model, patch, levels)\n            count[:, :, y : y + tile, xx : xx + tile] += 1\n    return out / count.clamp_min(1)\n\n\ndef load_image(path, device, multiple=1):\n    from PIL import Image\n    import numpy as np\n\n    img = Image.open(path).convert("RGB")\n    t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255.0)\n    # Models handle odd sizes internally. Keep all pixels for published benchmarks.\n    h, w = t.shape[1] // multiple * multiple, t.shape[2] // multiple * multiple\n    return t[:, :h, :w].unsqueeze(0).to(device)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    source = ap.add_mutually_exclusive_group(required=True)\n    source.add_argument("--ckpt")\n    source.add_argument("--ffdnet-weights", help="author\'s KAIR ffdnet_color.pth")\n    ap.add_argument("--author-protocol", action="store_true", help="FFDNet CBSD68: unclipped AWGN, seed 0 per image, uint8 output, full image")\n    ap.add_argument("--assert-psnr", type=float, help="require agreement within 0.5 dB")\n    ap.add_argument("--data", required=True, help="any ancestor of the DIV2K image directories")\n    ap.add_argument("--val-dir", help="skip the search and score this directory")\n    ap.add_argument("--out", default="results", help="benchmark.csv and figures/ go here")\n    ap.add_argument("--limit", type=int, default=20, help="validation images to score")\n    ap.add_argument("--notes", default="", help="e.g. smoke, final")\n    ap.add_argument("--method", default="", help="defaults to the model name in the checkpoint")\n    args = ap.parse_args()\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if args.ffdnet_weights:\n        ck = {"step": 0, "model": torch.load(args.ffdnet_weights, map_location=device, weights_only=True),\n              "extra": {"config": {"model": "ffdnet", "sigma": 25, "clip_noise": False}}}\n    else:\n        ck = torch.load(args.ckpt, map_location=device, weights_only=False)\n    cfg = ck["extra"]["config"]\n    method = args.method or cfg["model"]\n    sigma = cfg["sigma"] / 255.0\n\n    model = build_model(cfg["model"], **cfg.get("model_args", {})).to(device)\n    model.load_state_dict(ck["model"])\n    model.eval()\n    n_params = sum(p.numel() for p in model.parameters())\n    print(f"{method} @ step {ck[\'step\']}  params {n_params:,}  sigma {cfg[\'sigma\']}  device {device}")\n\n    val_dir = Path(args.val_dir) if args.val_dir else resolve_div2k(args.data)[1]\n    files = find_images(val_dir)[: args.limit]\n    if args.author_protocol and (cfg["model"] != "ffdnet" or len(files) != 68):\n        raise ValueError("author protocol requires FFDNet and all 68 clean CBSD68 images")\n    if not files:\n        raise ValueError("no evaluation images selected")\n\n    ins, outs, samples = [], [], []\n    for i, f in enumerate(files):\n        hr = load_image(f, device)\n        g = torch.Generator(device=device).manual_seed(i)\n        if args.author_protocol:\n            import numpy as np\n\n            noise = np.random.RandomState(0).normal(0, sigma, (*hr.shape[-2:], 3)).astype(np.float32)\n            noisy = hr + torch.from_numpy(noise).permute(2, 0, 1).unsqueeze(0).to(device)\n            with torch.no_grad():\n                pred = predict(model, noisy, sigma).clamp(0, 1).mul(255).round().div(255)\n        elif cfg.get("degradation") == "composite":\n            import numpy as np\n            from fcsg_net.degrade import degrade\n\n            lr, params = degrade(hr[0].cpu(), np.random.default_rng(i))\n            noisy = lr.unsqueeze(0).to(device)\n            pred = tiled_forward(model, noisy, sigma=params["noise_sigma"] / 255).clamp(0, 1)\n        else:\n            noisy = add_noise(hr, sigma, generator=g, clip=cfg.get("clip_noise", True))\n            pred = tiled_forward(model, noisy, sigma=sigma).clamp(0, 1)\n        p_in, p_out = psnr(noisy, hr, clip=False), psnr(pred, hr)\n        ins.append(p_in)\n        outs.append(p_out)\n        print(f"  {f.name}: {p_in:.2f} -> {p_out:.2f} dB")\n        if len(samples) < 2:\n            samples.append((noisy, pred, hr, p_in, p_out))\n\n    psnr_in, psnr_out = sum(ins) / len(ins), sum(outs) / len(outs)\n    print(f"\\nmean over {len(files)} images: {psnr_in:.2f} dB in -> {psnr_out:.2f} dB out "\n          f"(+{psnr_out - psnr_in:.2f})")\n\n    out_dir = Path(args.out)\n    log = CSVLog(out_dir / "benchmark.csv",\n                 ["method", "steps", "sigma", "psnr_in", "psnr_out", "gain", "params", "images", "notes"])\n    log.write(method=method, steps=ck["step"], sigma=cfg["sigma"],\n              psnr_in=f"{psnr_in:.3f}", psnr_out=f"{psnr_out:.3f}",\n              gain=f"{psnr_out - psnr_in:.3f}", params=n_params,\n              images=len(files), notes=args.notes)\n    print(f"appended row to {out_dir / \'benchmark.csv\'}")\n\n    save_qualitative(samples, out_dir / "figures" / "qualitative.png")\n    result = {"method": method, "steps": ck["step"], "images": len(files),\n              "psnr": psnr_out, "params": n_params, "notes": args.notes,\n              "protocol": "author-cbsd68" if args.author_protocol else "tiled-256-overlap32",\n              "source": args.ffdnet_weights or args.ckpt}\n    if args.assert_psnr is not None:\n        result.update(target=args.assert_psnr, gap=abs(psnr_out - args.assert_psnr),\n                      passed=abs(psnr_out - args.assert_psnr) <= 0.5)\n    (out_dir / "evaluation.json").write_text(json.dumps(result, indent=2) + "\\n")\n    if args.assert_psnr is not None:\n        assert result["passed"], result\n\n\ndef save_qualitative(samples, path, crop=320):\n    """Noisy / restored / clean strip. The figure a panel actually looks at."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    fig, axes = plt.subplots(len(samples), 3, figsize=(9, 3.1 * len(samples)), squeeze=False)\n    for r, (noisy, pred, hr, p_in, p_out) in enumerate(samples):\n        panels = [(noisy, f"noisy  {p_in:.2f} dB"), (pred, f"restored  {p_out:.2f} dB"), (hr, "ground truth")]\n        for c, (img, title) in enumerate(panels):\n            ax = axes[r][c]\n            ax.imshow(img[0, :, :crop, :crop].permute(1, 2, 0).cpu().numpy())\n            ax.set_title(title, fontsize=10)\n            ax.axis("off")\n    fig.tight_layout()\n    fig.savefig(path, dpi=130)\n    print(f"wrote {path}")\n\n\nif __name__ == "__main__":\n    main()\n', 'evaluation/plots.py': '"""Training curves from the CSV log.\n\n    python evaluation/plots.py --csv checkpoints/train_log.csv --out results/figures\n\nThe log is sparse: loss rows carry no PSNR and validation rows carry no loss, so\neach series is filtered independently rather than read as a table.\n"""\n\nimport argparse\nimport csv\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\n\nNAN = float("nan")\n\n\ndef series(rows, key):\n    return [(int(r["step"]), float(r[key])) for r in rows if r.get(key)]\n\n\ndef xy(pts, factor=5):\n    """Unzip to x/y lists, breaking the line across abnormally large step gaps.\n\n    A resumed session whose train_log.csv was never seeded leaves a hole in the\n    log. Plotted straight through, matplotlib interpolates a confident line over\n    steps that were never recorded, so insert NaN and let the gap read as one.\n    """\n    if len(pts) < 3:\n        return [p[0] for p in pts], [p[1] for p in pts]\n    deltas = sorted(b[0] - a[0] for a, b in zip(pts, pts[1:]))\n    med = deltas[len(deltas) // 2] or 1\n    xs, ys = [pts[0][0]], [pts[0][1]]\n    for a, b in zip(pts, pts[1:]):\n        if b[0] - a[0] > factor * med:\n            xs.append(NAN)\n            ys.append(NAN)\n        xs.append(b[0])\n        ys.append(b[1])\n    return xs, ys\n\n\ndef gaps(pts, factor=5):\n    """Step ranges with no logged data, for annotating."""\n    if len(pts) < 3:\n        return []\n    deltas = sorted(b[0] - a[0] for a, b in zip(pts, pts[1:]))\n    med = deltas[len(deltas) // 2] or 1\n    return [(a[0], b[0]) for a, b in zip(pts, pts[1:]) if b[0] - a[0] > factor * med]\n\n\ndef cumulative_hours(pts):\n    """secs restarts at zero every session, so accumulate for total GPU time.\n\n    Returns the cumulative series plus the steps where a new session began.\n    """\n    total = prev = 0.0\n    out, boundaries = [], []\n    for step, s in pts:\n        if s < prev:\n            total += prev\n            boundaries.append(step)\n        prev = s\n        out.append((step, (total + s) / 3600))\n    return out, boundaries\n\n\ndef save(fig, path, note=""):\n    fig.tight_layout()\n    fig.savefig(path, dpi=130)\n    plt.close(fig)\n    print(f"wrote {path}  {note}")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--csv", required=True)\n    ap.add_argument("--out", default="results/figures")\n    args = ap.parse_args()\n\n    with open(args.csv, newline="") as f:\n        rows = list(csv.DictReader(f))\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    loss = series(rows, "loss")\n    if loss:\n        fig, ax = plt.subplots(figsize=(6, 3.6))\n        ax.plot(*xy(loss), lw=1)\n        ax.set_yscale("log")\n        ax.set_xlabel("step")\n        ax.set_ylabel("Charbonnier loss")\n        ax.set_title("Training loss")\n        ax.grid(alpha=0.3)\n        save(fig, out / "loss_curve.png",\n             f"({len(loss)} points, {loss[0][1]:.4f} -> {loss[-1][1]:.4f})")\n\n    p_out = series(rows, "psnr_out")\n    p_in = series(rows, "psnr_in")\n    if p_out:\n        fig, ax = plt.subplots(figsize=(6, 3.6))\n        for lo, hi in gaps(p_out):\n            ax.axvspan(lo, hi, color="0.9", zorder=0)\n        ax.plot(*xy(p_out), marker="o", ms=3, label="restored")\n        if p_in:\n            base = sum(v for _, v in p_in) / len(p_in)\n            ax.axhline(base, ls="--", c="grey", label=f"noisy input ({base:.2f} dB)")\n        ax.set_xlabel("step")\n        ax.set_ylabel("PSNR (dB)")\n        ax.set_title("Validation PSNR")\n        ax.legend()\n        ax.grid(alpha=0.3)\n        best_step, best = max(p_out, key=lambda t: t[1])\n        save(fig, out / "psnr_curve.png", f"(best {best:.2f} dB at step {best_step})")\n\n    lr = series(rows, "lr")\n    if lr:\n        fig, ax = plt.subplots(figsize=(6, 3.6))\n        ax.plot(*xy(lr), lw=1.2, c="tab:orange")\n        ax.set_yscale("log")\n        ax.set_xlabel("step")\n        ax.set_ylabel("learning rate")\n        ax.set_title("Cosine schedule")\n        ax.grid(alpha=0.3)\n        save(fig, out / "lr_curve.png",\n             f"({lr[0][1]:.2e} -> {lr[-1][1]:.2e}, "\n             f"{lr[-1][1] / max(lr[0][1], 1e-12) * 100:.1f}% of peak at the end)")\n\n    secs = series(rows, "secs")\n    if secs:\n        hours, boundaries = cumulative_hours(secs)\n        # Steps advanced during unlogged sessions, so total steps over total\n        # logged hours would overstate the rate. Take the median in-session slope.\n        slopes = sorted((b[0] - a[0]) / ((b[1] - a[1]) * 3600)\n                        for a, b in zip(hours, hours[1:]) if b[1] > a[1] and b[0] > a[0])\n        rate = slopes[len(slopes) // 2] if slopes else float("nan")\n        ys, xs = xy([(s, h) for s, h in hours])\n\n        fig, ax = plt.subplots(figsize=(6, 3.6))\n        ax.plot(xs, ys, lw=1.4, c="tab:green")\n        for step in boundaries:\n            ax.axvline(next(h for s, h in hours if s == step), ls=":", c="grey", lw=1)\n        ax.set_xlabel("logged GPU time (hours)")\n        ax.set_ylabel("step")\n        ax.set_title(f"Throughput, {rate:.2f} steps/s (dotted = session restart)")\n        ax.grid(alpha=0.3)\n        missing = sum(hi - lo for lo, hi in gaps([(s, h) for s, h in hours]))\n        note = f"({hours[-1][0]:,} steps, {hours[-1][1]:.1f} h logged, {rate:.2f} steps/s)"\n        if missing:\n            note += f"  WARNING: {missing:,} steps ran in unlogged sessions (~{missing / rate / 3600:.1f} h)"\n        save(fig, out / "throughput.png", note)\n\n    if not loss and not p_out:\n        raise SystemExit(f"nothing to plot: {args.csv} has no loss or psnr rows")\n\n\nif __name__ == "__main__":\n    main()\n', 'evaluation/run_milestones.py': '"""Run M1\'s FFDNet benchmark, M2 checks, and the D2 export on Kaggle/Colab."""\n\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport urllib.request\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path[:0] = [str(ROOT), str(ROOT / "src")]\n\n\ndef download(url, path):\n    urllib.request.urlretrieve(url, path)\n    return path\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out", required=True, help="new output directory")\n    args = ap.parse_args()\n    import numpy as np\n    import PIL\n    import torch\n    from fcsg_net.degrade import snapshot\n    from fcsg_net.utils import resolve_div2k\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable the Kaggle/Colab GPU before running milestones")\n    out = Path(args.out).resolve()\n    out.mkdir(parents=True, exist_ok=False)\n    environment = {"python": sys.version, "torch": torch.__version__,\n                   "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(),\n                   "numpy": np.__version__, "pillow": PIL.__version__}\n    (out / "environment.json").write_text(json.dumps(environment, indent=2) + "\\n")\n    weights_url = "https://github.com/cszn/KAIR/releases/download/v1.0/ffdnet_color.pth"\n    weights = download(weights_url, out / "ffdnet_color.pth")\n    images = out / "cbsd68"\n    images.mkdir()\n    request = urllib.request.Request(\n        "https://api.github.com/repos/cszn/FFDNet/contents/testsets/CBSD68",\n        headers={"User-Agent": "FCSG-Net-milestone-check"})\n    with urllib.request.urlopen(request, timeout=60) as response:\n        files = [f for f in json.load(response) if f["name"].lower().endswith(".png")]\n    if len(files) != 68:\n        raise ValueError(f"expected 68 author-provided CBSD68 PNGs, found {len(files)}")\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        list(pool.map(lambda f: download(f["download_url"], images / f["name"]), files))\n    provenance = {"weights_url": weights_url, "weights_sha256": sha256(weights),\n                  "weight_source": "author pretrained; not a from-scratch reproduction",\n                  "images": [{"name": f["name"], "url": f["download_url"],\n                              "sha256": sha256(images / f["name"])} for f in files]}\n    (out / "phase1_provenance.json").write_text(json.dumps(provenance, indent=2) + "\\n")\n    subprocess.run([sys.executable, str(ROOT / "evaluation/eval.py"),\n                    "--ffdnet-weights", str(weights), "--data", str(images),\n                    "--val-dir", str(images), "--limit", "68", "--author-protocol",\n                    "--assert-psnr", "31.21", "--out", str(out / "ffdnet"),\n                    "--notes", "phase1-cbsd68-author-pretrained-unclipped"], check=True)\n    train, val = resolve_div2k(args.data)\n    if train == val:\n        raise ValueError("milestone data must have separate training and validation splits")\n    subprocess.run([sys.executable, str(ROOT / "evaluation/checks.py"),\n                    "--data", str(train), "--out", str(out / "phase2_checks.json")], check=True)\n    result = snapshot(train, out / "d2", count=5000)\n    assert len(list((out / "d2/hr").glob("*.png"))) == 5000\n    assert len(list((out / "d2/lr").glob("*.png"))) == 5000\n    archive = shutil.make_archive(str(out / "d2_snapshot"), "zip", out / "d2")\n    result.update(archive=Path(archive).name, archive_sha256=sha256(archive),\n                  manifest_sha256=sha256(out / "d2/manifest.csv"))\n    (out / "phase2_snapshot.json").write_text(json.dumps(result, indent=2) + "\\n")\n    print("FFDNet benchmark, Phase 2 checks, and 5,000-pair D2 snapshot passed.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'configs/dncnn.toml': '# DnCNN baseline\nmodel = "dncnn"\nsigma = 25\npatch = 128\nbatch = 16\nsteps = 300000\nlr = 2e-4\nlr_min = 1e-6\nweight_decay = 0.0\namp = true\nworkers = 4\nlog_every = 50\nval_every = 2000\nckpt_every = 2000\nval_images = 16\nmax_hours = 11.0\n', 'configs/fcsg.toml': '# Phase 2 integration and overfit checks. Phase 3 adds the research losses.\nmodel = "fcsg"\ndegradation = "composite"\nsigma = 25\npatch = 128\nbatch = 16\nsteps = 300000\nlr = 2e-4\nlr_min = 1e-6\nweight_decay = 0.0\namp = true\nworkers = 4\nlog_every = 50\nval_every = 2000\nckpt_every = 2000\nval_images = 16\nmax_hours = 11.0\n', 'configs/ffdnet.toml': 'model = "ffdnet"\nsigma = 25\nclip_noise = false\npatch = 128\nbatch = 16\nsteps = 300000\nlr = 2e-4\nlr_min = 1e-6\nweight_decay = 0.0\namp = true\nworkers = 4\nlog_every = 50\nval_every = 2000\nckpt_every = 2000\nval_images = 16\nmax_hours = 11.0\n'}
assert hashlib.sha256(json.dumps(FILES, sort_keys=True).encode()).hexdigest() == '48ef7806ff38a153be54b0a0e3374b43c97d192a833d0a9812297064cbb86985'
for name, source in FILES.items():
    path = REPO / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
os.chdir(REPO)
print('Prepared', len(FILES), 'source files in', REPO)


Prepared 18 source files in /kaggle/working/fcsg-milestone-code


## Run the benchmark, model checks, and D2 export

The FFDNet gate requires 31.21 dB ± 0.5 dB on all 68 clean CBSD68 images.
The architecture checks enforce reconstruction, routing gradients, parameter
and compute budgets, a 4 GB CUDA allocator limit, and a 200-step overfit test.
The final export contains 5,000 pairs and the parameters that generated each pair.


In [6]:
subprocess.run([
    sys.executable, '-u', 'evaluation/run_milestones.py',
    '--data', str(DATA), '--out', str(OUT),
], check=True)


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.34363237..1.3727118].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.35847676..1.420182].


ffdnet @ step 0  params 852,108  sigma 25  device cuda
  101085.png: 20.18 -> 28.14 dB
  101087.png: 20.18 -> 32.23 dB
  102061.png: 20.18 -> 32.12 dB
  103070.png: 20.18 -> 32.81 dB
  105025.png: 20.18 -> 29.91 dB
  106024.png: 20.18 -> 34.85 dB
  108005.png: 20.18 -> 30.76 dB
  108070.png: 20.18 -> 28.66 dB
  108082.png: 20.18 -> 31.28 dB
  109053.png: 20.18 -> 31.77 dB
  119082.png: 20.18 -> 31.31 dB
  12084.png: 20.18 -> 29.96 dB
  123074.png: 20.18 -> 31.82 dB
  126007.png: 20.18 -> 32.76 dB
  130026.png: 20.18 -> 29.74 dB
  134035.png: 20.18 -> 30.54 dB
  14037.png: 20.18 -> 34.50 dB
  143090.png: 20.18 -> 34.72 dB
  145086.png: 20.18 -> 30.45 dB
  147091.png: 20.18 -> 30.97 dB
  148026.png: 20.18 -> 28.88 dB
  148089.png: 20.18 -> 29.40 dB
  157055.png: 20.18 -> 31.13 dB
  159008.png: 20.18 -> 31.03 dB
  160068.png: 20.18 -> 31.87 dB
  16077.png: 20.18 -> 30.55 dB
  163085.png: 20.18 -> 31.98 dB
  167062.png: 20.18 -> 34.92 dB
  167083.png: 20.18 -> 27.49 dB
  170057.png: 20.18 

/kaggle/working/fcsg-milestone-code/src/fcsg_net/data.py:42: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div(255.0)


overfit 50/200 mse=0.023472
overfit 100/200 mse=0.016215
overfit 150/200 mse=0.009905
overfit 200/200 mse=0.004979
{
  "device": "Tesla T4",
  "torch": "2.10.0+cu128",
  "seed": 7,
  "reconstruction_max_error": 5.960464477539062e-07,
  "frequency_units_check": true,
  "ffdnet_noise_map_check": true,
  "degradation_samples_per_second": 244.3658345009721,
  "degradation_reproducible": true,
  "parameters": 97088,
  "peak_allocated_bytes": 536057856,
  "peak_reserved_bytes": 597688320,
  "conv_gmacs": 4.664494143,
  "gflops_with_fft_estimate": 9.391902846,
  "flop_convention": "2 FLOPs/MAC; thop ops plus conservative FFT estimate; elementwise ops excluded",
  "forward_backward_routing_checks": true,
  "amp_odd_size_check": true,
  "checkpoint_resume_check": true,
  "overfit_passed": true,
  "passed": true
}
overfit MSE 0.026092 -> 0.004932
D2 snapshot 250/5000
D2 snapshot 500/5000
D2 snapshot 750/5000
D2 snapshot 1000/5000
D2 snapshot 1250/5000
D2 snapshot 1500/5000
D2 snapshot 1750/5000


CompletedProcess(args=['/usr/bin/python3', '-u', 'evaluation/run_milestones.py', '--data', '/kaggle/input', '--out', '/kaggle/working/fcsg-milestones-20260923-102230'], returncode=0)

## Collect the results

Download both archives. Keep the printed GPU identity with the results.


In [7]:
import zipfile
from IPython.display import FileLink, display

manifest = {name: hashlib.sha256(source.encode()).hexdigest() for name, source in FILES.items()}
(OUT / 'source_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
for name in ('ffdnet/evaluation.json', 'phase2_checks.json', 'phase2_snapshot.json'):
    result = json.loads((OUT / name).read_text())
    if 'overfit' in result:
        result['overfit'].pop('losses', None)
    print(name, json.dumps(result, indent=2))
archive = OUT / 'milestone_evidence.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    paths = list(OUT.glob('*.json')) + list((OUT / 'ffdnet').rglob('*')) + [OUT / 'd2/manifest.csv']
    for path in sorted(paths):
        if path.is_file():
            z.write(path, path.relative_to(OUT))
display(FileLink(str(archive)))
display(FileLink(str(OUT / 'd2_snapshot.zip')))
print('The runtime checks passed. Preserve these results before ending the session.')


ffdnet/evaluation.json {
  "method": "ffdnet",
  "steps": 0,
  "images": 68,
  "psnr": 31.21981247733621,
  "params": 852108,
  "notes": "phase1-cbsd68-author-pretrained-unclipped",
  "protocol": "author-cbsd68",
  "source": "/kaggle/working/fcsg-milestones-20260923-102230/ffdnet_color.pth",
  "target": 31.21,
  "gap": 0.009812477336208048,
  "passed": true
}
phase2_checks.json {
  "device": "Tesla T4",
  "torch": "2.10.0+cu128",
  "seed": 7,
  "reconstruction_max_error": 5.960464477539062e-07,
  "frequency_units_check": true,
  "ffdnet_noise_map_check": true,
  "degradation_samples_per_second": 244.3658345009721,
  "degradation_reproducible": true,
  "parameters": 97088,
  "peak_allocated_bytes": 536057856,
  "peak_reserved_bytes": 597688320,
  "conv_gmacs": 4.664494143,
  "gflops_with_fft_estimate": 9.391902846,
  "flop_convention": "2 FLOPs/MAC; thop ops plus conservative FFT estimate; elementwise ops excluded",
  "forward_backward_routing_checks": true,
  "amp_odd_size_check": true

/kaggle/working/fcsg-milestones-20260923-102230/milestone_evidence.zip

/kaggle/working/fcsg-milestones-20260923-102230/d2_snapshot.zip

The runtime checks passed. Preserve these results before ending the session.
